# 02 — Feature Engineering

Builds the analytical feature tables used downstream:

| Table | Grain | Purpose |
|-------|-------|---------|
| `features_orders` | order | Cycle times, delay, temporal buckets |
| `features_customers` | customer | RFM segmentation + satisfaction |
| `features_products` | product | Revenue, price, freight, late rate |
| `features_sellers` | seller | Revenue, delivery, late rate |

All logic lives in `src/features.py` — this notebook just runs and inspects.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import DATA_PROCESSED
from src import features as F
from src.viz import PALETTE, save
from src.config import DATA_PROCESSED

sns.set_style("whitegrid")

In [ ]:
required = [
    "customers.parquet",
    "orders.parquet",
    "order_items.parquet",
    "products.parquet",
    "sellers.parquet",
    "category_translation.parquet",
]
missing = [f for f in required if not (DATA_PROCESSED / f).exists()]
if missing:
    raise RuntimeError(f"Missing intermediate files: {missing}. Run ETL first.")
print("All prerequisites present.")

In [ ]:
feature_tables = F.build_all_features()
for name, df in feature_tables.items():
    print(f"{name:12s} {df.shape}")

In [ ]:
orders_f = feature_tables['orders']

orders_f[[
    'days_to_approval', 'days_to_carrier',
    'days_to_delivery', 'days_estimated',
    'delivery_delay_days'
]].describe().round(2)

In [ ]:
delivered = orders_f[orders_f['days_to_delivery'].notna()]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(delivered['days_to_delivery'].clip(0, 60),
             bins=40, color=PALETTE['primary'], edgecolor='white')
axes[0].set_title('Actual Delivery Days (clipped at 60)')
axes[0].set_xlabel('Days')

axes[1].hist(delivered['delivery_delay_days'].clip(-30, 30),
             bins=40, color=PALETTE['accent'], edgecolor='white')
axes[1].axvline(0, color='black', linewidth=1)
axes[1].set_title('Delay vs Estimate (negative = early)')
axes[1].set_xlabel('Days')

save(fig, '02_delivery_features')
plt.show()

print(f"Late rate: {delivered['is_late'].mean():.1%}")

In [ ]:
rfm = feature_tables['customers']

print(rfm['segment'].value_counts())
print()
print(rfm[['recency_days', 'frequency', 'monetary']].describe().round(2))

fig, ax = plt.subplots(figsize=(10, 5))
sns.countplot(
    data=rfm, x='segment',
    order=rfm['segment'].value_counts().index,
    palette='viridis', ax=ax, hue='segment', legend=False
)
ax.set_title('Customer Segments (RFM)')
ax.set_ylabel('Customers')
save(fig, '02_rfm_segments')
plt.show()

Sanity checks:
- Champions should have low recency, high frequency, high monetary
- Lost should be the opposite
- Segment counts should be reasonable

In [ ]:
rfm.groupby('segment')[
    ['recency_days', 'frequency', 'monetary']
].mean().round(2).sort_values('monetary', ascending=False)

In [ ]:
products = feature_tables['products']
sellers = feature_tables['sellers']

print("Top 10 products by revenue:")
display(
    products.sort_values('revenue', ascending=False)
    .head(10)[['product_id', 'category', 'orders', 'revenue', 'late_rate']]
    .round(2)
)

print("\nSeller concentration:")
top5 = sellers.nlargest(5, 'revenue')['revenue'].sum()
total = sellers['revenue'].sum()
print(f"Top 5 sellers -> {top5 / total:.1%} of total revenue")

## Outputs

All four feature tables are saved as Parquet in `data/processed/`. The business analysis notebook loads these directly.